# Practice 49 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np


## Level 1 — subscription renewals

Use the subscriptions data below. Plan durations: `'monthly'` = 1 month, `'annual'` = 12 months.

- Add `trial_end`: 14 days after `signup_date`. Vectorized — no `.apply()`.
- Add `renewal_date`: `signup_date` + plan duration. The offset varies per row, so use `.apply(axis=1)`.
- Add `days_until_renewal`: number of days from `signup_date` to `renewal_date` (integer).
- How many customers renew in 2025? Filter using `.dt.year`.
- What is the mean `days_until_renewal` per plan type? One groupby.

In [17]:
subs = pd.DataFrame({
    'customer':    ['Alice', 'Bob', 'Carol', 'Dave', 'Eve', 'Frank'],
    'signup_date': pd.to_datetime(['2024-01-15', '2024-02-28', '2024-08-10',
                                   '2024-11-01', '2024-12-20', '2025-01-05']),
    'plan':        ['monthly', 'annual', 'monthly', 'annual', 'monthly', 'annual'],
})

# Your code here
subs['trial_end'] = subs['signup_date'] + pd.DateOffset(days = 14)

subs['renewal_date'] = subs.apply(lambda row: row['signup_date'] + pd.DateOffset(months=1) if row['plan'] == 'monthly'
                                  else row['signup_date'] + pd.DateOffset(months = 12), axis = 1)

subs['days_until_renewal'] = (subs['renewal_date'] - subs['signup_date']).dt.days

(subs['renewal_date'].dt.year == 2025).sum()

subs.groupby('plan')['days_until_renewal'].mean()


plan
annual     365.333333
monthly     31.000000
Name: days_until_renewal, dtype: float64

In [14]:
subs

,customer,signup_date,plan,trial_end,renewal_date,days_until_renewal
0,Alice,2024-01-15,monthly,2024-01-29,2024-02-15,31
1,Bob,2024-02-28,annual,2024-03-13,2025-02-28,366
2,Carol,2024-08-10,monthly,2024-08-24,2024-09-10,31
3,Dave,2024-11-01,annual,2024-11-15,2025-11-01,365
4,Eve,2024-12-20,monthly,2025-01-03,2025-01-20,31
5,Frank,2025-01-05,annual,2025-01-19,2026-01-05,365


---
## Level 2 — quarterly prices

The `prices` DataFrame has a quarterly `DatetimeIndex`. Use it to answer the questions below — no step-by-step, figure out the approach.

- Add `prev_quarter`: price from the previous quarter.
- Add `qoq_pct`: quarter-over-quarter % change, rounded to 1 decimal place.
- Add `year_ago`: price from 4 quarters ago. What does the first year of data look like for this column?
- Add `yoy_pct`: year-over-year % change.
- Which quarter had the largest YoY gain? The largest YoY drop?
- Use `shift(2, freq='QS')` to create a copy with the index shifted forward 2 quarters. What would you use this for?

In [47]:
prices = pd.DataFrame(
    {'price': [100.0, 104.0, 98.0, 110.0, 108.0, 115.0, 112.0, 121.0]},
    index=pd.date_range('2023-01', periods=8, freq='QS')
)

# Your code here

#prices['prev_quarter'] = prices['price'].shift(1, freq = 'QS')
prices['prev_quarter'] = prices['price'].shift(1)

prices['qoq_pct'] = round((prices['price'] - prices['prev_quarter'])/prices['prev_quarter'],1)

prices['year_ago'] = prices['price'].shift(4)
prices['yoy_pct'] = round((prices['price'] - prices['year_ago'])/prices['year_ago'],2)

prices['yoy_pct'].idxmax()
prices['yoy_pct'].idxmin()

new = prices.copy()
new = new.shift(2, freq = 'QS')

### use for half year estimate 

---
## Level 3 — global trades

All timestamps are UTC. Each trade belongs to an exchange with its own timezone and market hours:

| Exchange | Timezone | Market hours (local) |
|----------|----------|---------------------|
| NYSE | `US/Eastern` | 9:30–16:00 |
| LSE | `Europe/London` | 8:00–16:30 |
| TSE | `Asia/Tokyo` | 9:00–15:30 |

Write a `.pipe()` chain:
- **`localize(df)`** — localize `timestamp_utc` to UTC. Add `time_local`: the timestamp converted to each row's exchange timezone. Since each row may have a different timezone, use `.apply()` with `dt.tz_convert()`. Add `local_hour` from `time_local`.
- **`classify(df)`** — add `is_market_hours` using the table above (hint: `np.select` with per-exchange hour conditions). Add `amount_tier`: `'small'` (< 40k), `'large'` (≥ 40k).

After the chain:
- What fraction of trades occurred during market hours?
- Which exchange had the highest mean `amount` during market hours?
- Among `'large'` trades, how many were outside market hours?

In [78]:
trades = pd.DataFrame({
    'trade_id':      ['T001', 'T002', 'T003', 'T004', 'T005', 'T006', 'T007', 'T008'],
    'timestamp_utc': pd.to_datetime([
        '2024-03-15 14:00', '2024-03-15 08:30', '2024-03-15 01:00',
        '2024-03-15 20:00', '2024-03-15 10:00', '2024-03-15 05:00',
        '2024-03-15 16:30', '2024-03-15 12:00',
    ]),
    'exchange': ['NYSE', 'LSE', 'TSE', 'NYSE', 'LSE', 'TSE', 'NYSE', 'LSE'],
    'amount':   [50000, 32000, 78000, 25000, 91000, 43000, 67000, 55000],
})

# Your code here

def localize(df):
    df['timestamp_utc'] = df['timestamp_utc'].dt.tz_localize('UTC')
    df['time_local']  = df.apply(lambda row: row['timestamp_utc'].tz_convert('US/Eastern') if row['exchange'] == 'NYSE'
                                 else row['timestamp_utc'].tz_convert('Europe/London') if row['exchange'] == 'LSE'
                                 else row['timestamp_utc'].tz_convert('Asia/Tokyo'), axis = 1)
    df['local_hour'] = df.apply(
    lambda row: row['timestamp_utc'].tz_convert('US/Eastern').hour  if row['exchange'] == 'NYSE'
           else row['timestamp_utc'].tz_convert('Europe/London').hour if row['exchange'] == 'LSE'
           else row['timestamp_utc'].tz_convert('Asia/Tokyo').hour,
    axis=1)
    return df

def classify(df):
    conds = [df['exchange']=='NYSE', df['exchange'] == 'LSE', df['exchange'] == 'TSE']
    chos_start = [9.5,8,9]
    chos_end = [16,16.5,15.5]
    df['start'] = np.select(conds, chos_start, default = 0)
    df['end'] = np.select(conds, chos_end, default = 0)
    df['is_market_hour'] = (df['local_hour']>= df['start']) & (df['local_hour']<= df['end'])
    df['amonut_tier']  = pd.cut(df['amount'], bins = [0, 40000,np.inf], labels=['small','large'],right=False)
    return df 



After the chain:
- What fraction of trades occurred during market hours?
- Which exchange had the highest mean `amount` during market hours?
- Among `'large'` trades, how many were outside market hours?

In [83]:

res = trades.copy().pipe(localize).pipe(classify)

res['is_market_hour'].mean()

res.groupby('exchange')['amount'].mean().idxmax()

1-res.loc[res['amonut_tier']=='large','is_market_hour'].mean()


np.float64(0.0)